# LeetCode 2113 — Elements in Array After Removing and Replacing Elements bằng Python

Link đề: https://leetcode.com/problems/elements-in-array-after-removing-and-replacing-elements/ · Độ khó: **Medium** · Chủ đề: Array

> **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def elementInNums(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        ...
```

Lộ trình:
1. **Mô phỏng từng phút** bằng `deque` cho mỗi truy vấn — `O(time · n)` mỗi truy vấn.
2. **Mô phỏng một chu kỳ** `2n` phút, lưu mọi trạng thái, trả lời bằng `time mod 2n` — `O(n²)` tiền xử lý, `O(1)` mỗi truy vấn.
3. **Công thức đóng** theo pha "co" / "giãn" — `O(1)` mỗi truy vấn, `O(1)` bộ nhớ phụ (**nộp bài**).
4. **Biểu diễn mỗi trạng thái bằng một đoạn `[lo, hi)` của mảng gốc** — cách nhìn thống nhất cho cả hai pha.

## 1. Mục tiêu học

1. Nhận ra quá trình **tuần hoàn** với chu kỳ `2n` (n phút xóa đầu, n phút nối lại).
2. Mô tả mỗi trạng thái như một **đoạn liên tiếp** của mảng gốc: pha co là hậu tố `nums[t:]`, pha giãn là tiền tố `nums[:t−n]`.
3. Trả lời truy vấn bằng công thức `O(1)` và xử lý đúng biên (`t = 0`, `t = n` mảng rỗng, chỉ số vượt độ dài).
4. Kiểm chứng công thức bằng mô phỏng thật trên mọi `t` của vài chu kỳ.
5. Mở rộng sang các quá trình tuần hoàn khác (xoay vòng, xóa cuối…).

## 2. Đề bài & ví dụ

Ban đầu (phút 0) mảng `nums` nguyên vẹn. Mỗi phút, phần tử **trái nhất** bị xóa cho tới khi mảng rỗng. Sau đó mỗi phút **nối** một phần tử vào cuối, theo đúng thứ tự đã bị xóa, cho tới khi mảng trở về như ban đầu. Quá trình lặp mãi.

Ví dụ `[0,1,2]`: `[0,1,2] → [1,2] → [2] → [] → [0] → [0,1] → [0,1,2] → [1,2] → …`

Truy vấn `[time, index]`: trả về `nums[index]` **của mảng tại phút `time`** nếu `index` nhỏ hơn độ dài mảng lúc đó, ngược lại `-1`.

**Ví dụ 1:** `nums = [0,1,2]`, `queries = [[0,2],[2,0],[3,2],[5,0]]` → `[2,2,-1,0]`.

**Ví dụ 2:** `nums = [2]`, `queries = [[0,0],[1,0],[2,0],[3,0]]` → `[2,-1,2,-1]`.

**Ràng buộc:** `1 ≤ n ≤ 100`, `0 ≤ nums[i] ≤ 100`, `1 ≤ len(queries) ≤ 10⁵`, `0 ≤ time ≤ 10⁵`, `0 ≤ index < n`.

In [1]:
import random
import time as _time
from collections import deque
from typing import List

examples = [(([0, 1, 2], [[0, 2], [2, 0], [3, 2], [5, 0]]), [2, 2, -1, 0]),
            (([2], [[0, 0], [1, 0], [2, 0], [3, 0]]), [2, -1, 2, -1])]
for (nums, qs), want in examples:
    print(nums, qs, "->", want)

[0, 1, 2] [[0, 2], [2, 0], [3, 2], [5, 0]] -> [2, 2, -1, 0]
[2] [[0, 0], [1, 0], [2, 0], [3, 0]] -> [2, -1, 2, -1]


## 3. Kiến thức nền

### 3.1 `deque` cho thao tác hai đầu

`popleft()` và `append()` đều `O(1)` — mô phỏng đúng "xóa trái nhất" và "nối vào cuối".

In [2]:
d = deque([0, 1, 2]); removed = []
removed.append(d.popleft()); print(d, removed)
d.append(removed.pop(0)); print(d)

deque([1, 2]) [0]
deque([1, 2, 0])


### 3.2 Phép chia lấy dư cho quá trình tuần hoàn

Nếu trạng thái ở phút `t` bằng trạng thái ở phút `t + P` với mọi `t`, thì chỉ cần biết `t mod P`.

### 3.3 Lát cắt Python

`nums[t:]` (hậu tố) và `nums[:k]` (tiền tố) — mỗi trạng thái của bài là một trong hai dạng này.

In [3]:
nums = [0, 1, 2]; n = len(nums)
for t in range(2 * n + 1):
    state = nums[t:] if t <= n else nums[:t - n]
    print(f"phút {t}: {state}")

phút 0: [0, 1, 2]
phút 1: [1, 2]
phút 2: [2]
phút 3: []
phút 4: [0]
phút 5: [0, 1]
phút 6: [0, 1, 2]


## 4. Ý tưởng nền tảng / trực giác

Với `n = 3`:

| phút `t` | `t mod 6` | pha | mảng | độ dài | phần tử ở chỉ số `idx` |
|---|---|---|---|---|---|
| 0 | 0 | co | [0,1,2] | 3 | `nums[0 + idx]` |
| 1 | 1 | co | [1,2] | 2 | `nums[1 + idx]` |
| 2 | 2 | co | [2] | 1 | `nums[2 + idx]` |
| 3 | 3 | — | [] | 0 | -1 |
| 4 | 4 | giãn | [0] | 1 | `nums[idx]` |
| 5 | 5 | giãn | [0,1] | 2 | `nums[idx]` |
| 6 | 0 | co | [0,1,2] | 3 | … |

Quy luật với `t' = t mod 2n`:
- `t' < n` (co): mảng là `nums[t':]`, độ dài `n − t'`; phần tử chỉ số `idx` là `nums[t' + idx]` nếu `idx < n − t'`.
- `t' ≥ n` (giãn, gồm cả `t' = n` rỗng): mảng là `nums[:t' − n]`, độ dài `t' − n`; phần tử là `nums[idx]` nếu `idx < t' − n`.

In [4]:
nums = [0, 1, 2]
for t, idx in [[0, 2], [2, 0], [3, 2], [5, 0], [7, 1], [10, 0]]:
    tp = t % 6
    if tp < 3:
        phase, L, val = "co", 3 - tp, nums[tp + idx] if idx < 3 - tp else -1
    else:
        phase, L, val = "giãn", tp - 3, nums[idx] if idx < tp - 3 else -1
    print(f"t = {t:>2} (t' = {tp}), idx = {idx}: pha {phase}, độ dài {L} -> {val}")

t =  0 (t' = 0), idx = 2: pha co, độ dài 3 -> 2
t =  2 (t' = 2), idx = 0: pha co, độ dài 1 -> 2
t =  3 (t' = 3), idx = 2: pha giãn, độ dài 0 -> -1
t =  5 (t' = 5), idx = 0: pha giãn, độ dài 2 -> 0
t =  7 (t' = 1), idx = 1: pha co, độ dài 2 -> 2
t = 10 (t' = 4), idx = 0: pha giãn, độ dài 1 -> 0


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng phút cho mỗi truy vấn

Điểm mấu chốt khi mô phỏng: phải nhớ **đang ở pha nào** (xóa hay nối) bằng một biến trạng thái `shrinking`; không suy ra pha từ độ dài mảng (xem Lỗi 5).

In [5]:
def element_simulate(nums, queries):
    n = len(nums); out = []
    for t, idx in queries:
        d = deque(nums); removed = deque(); shrinking = True
        for _ in range(t):
            if shrinking:
                removed.append(d.popleft())
                if not d:
                    shrinking = False          # đã rỗng: chuyển sang pha nối
            else:
                d.append(removed.popleft())
                if len(d) == n:
                    shrinking = True           # đã khôi phục: lại xóa
        out.append(d[idx] if idx < len(d) else -1)
    return out

for (nums, qs), want in examples:
    print(element_simulate(nums, qs), "kỳ vọng", want)

[2, 2, -1, 0] kỳ vọng [2, 2, -1, 0]
[2, -1, 2, -1] kỳ vọng [2, -1, 2, -1]


**Độ phức tạp:** mỗi truy vấn `O(time + n)` ⇒ tổng tới `10⁵ · 10⁵ = 10¹⁰` — quá chậm. Space `O(n)`.

**Nút thắt:** trạng thái lặp lại sau `2n` phút; mô phỏng lại từ đầu cho mỗi truy vấn là lãng phí.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (chu kỳ):** sau đúng `n` phút mảng rỗng, sau `n` phút nữa mảng khôi phục ⇒ chu kỳ `P = 2n`. Trạng thái phút `t` = trạng thái phút `t mod 2n`.

**Insight 2 (dạng đoạn):** trong pha co, các phần tử bị xóa từ trái nên phần còn lại là **hậu tố** `nums[t':]`. Trong pha giãn, các phần tử được nối lại theo thứ tự bị xóa (`nums[0], nums[1], …`) nên mảng là **tiền tố** `nums[:t'−n]`.

**Quy tắc:**
- *Nếu* `t' < n` *thì* độ dài `n − t'`, phần tử `nums[t' + idx]`.
- *Ngược lại* độ dài `t' − n`, phần tử `nums[idx]`.
- *Nếu* `idx ≥ độ dài` *thì* `-1`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng một chu kỳ, lưu mọi trạng thái

In [6]:
def element_precompute(nums, queries):
    n = len(nums); states = []
    d = deque(nums); removed = deque(); shrinking = True
    for _ in range(2 * n):
        states.append(list(d))
        if shrinking:
            removed.append(d.popleft())
            if not d: shrinking = False
        else:
            d.append(removed.popleft())
            if len(d) == n: shrinking = True
    out = []
    for t, idx in queries:
        s = states[t % (2 * n)]
        out.append(s[idx] if idx < len(s) else -1)
    return out

for (nums, qs), want in examples:
    print(element_precompute(nums, qs), "kỳ vọng", want)

[2, 2, -1, 0] kỳ vọng [2, 2, -1, 0]
[2, -1, 2, -1] kỳ vọng [2, -1, 2, -1]


**Độ phức tạp:** tiền xử lý `O(n²)` (`2n` trạng thái × `O(n)` sao chép) ⇒ `2·10⁴` với `n = 100`; mỗi truy vấn `O(1)`. Space `O(n²)`. Không cần suy ra công thức — an toàn khi luật phức tạp hơn.

### 7.2 Cách 3 — Công thức đóng (code nộp)

In [7]:
def element_formula(nums, queries):
    n = len(nums); P = 2 * n; out = []
    for t, idx in queries:
        tp = t % P
        if tp < n:                                  # pha co: nums[tp:]
            out.append(nums[tp + idx] if idx < n - tp else -1)
        else:                                       # pha giãn: nums[:tp - n]
            out.append(nums[idx] if idx < tp - n else -1)
    return out

for (nums, qs), want in examples:
    print(element_formula(nums, qs), "kỳ vọng", want)

[2, 2, -1, 0] kỳ vọng [2, 2, -1, 0]
[2, -1, 2, -1] kỳ vọng [2, -1, 2, -1]


Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(1)` mỗi truy vấn ⇒ `O(q)` tổng, `O(1)` bộ nhớ phụ.

### 7.3 Cách 4 — Trạng thái là đoạn `[lo, hi)` của mảng gốc

Viết hàm `segment(t)` trả về `(lo, hi)`: pha co `(t', n)`, pha giãn `(0, t' − n)`. Truy vấn: `nums[lo + idx]` nếu `lo + idx < hi`. Cách này gom hai nhánh vào một biểu thức và dễ mở rộng (ví dụ biến thể xóa từ phải).

In [8]:
def segment(n, t):
    tp = t % (2 * n)
    return (tp, n) if tp < n else (0, tp - n)

def element_segment(nums, queries):
    n = len(nums); out = []
    for t, idx in queries:
        lo, hi = segment(n, t)
        out.append(nums[lo + idx] if lo + idx < hi else -1)
    return out

for (nums, qs), want in examples:
    print(element_segment(nums, qs), "kỳ vọng", want)
print("các đoạn của n = 3:", [segment(3, t) for t in range(7)])

[2, 2, -1, 0] kỳ vọng [2, 2, -1, 0]
[2, -1, 2, -1] kỳ vọng [2, -1, 2, -1]
các đoạn của n = 3: [(0, 3), (1, 3), (2, 3), (0, 0), (0, 1), (0, 2), (0, 3)]


## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1: in `t`, `t mod 2n`, pha, độ dài và chỉ số trong mảng gốc.

In [9]:
def trace_formula(nums, queries):
    n = len(nums); P = 2 * n
    for t, idx in queries:
        tp = t % P
        if tp < n:
            L = n - tp; src = tp + idx; phase = f"co: nums[{tp}:]"
        else:
            L = tp - n; src = idx; phase = f"giãn: nums[:{tp - n}]"
        ans = nums[src] if idx < L else -1
        print(f"t = {t:>3}, idx = {idx}: t mod {P} = {tp}, {phase}, độ dài {L} -> "
              + (f"nums[{src}] = {ans}" if idx < L else "-1 (vượt độ dài)"))

trace_formula([0, 1, 2], [[0, 2], [2, 0], [3, 2], [5, 0], [6, 2], [100000, 1]])

t =   0, idx = 2: t mod 6 = 0, co: nums[0:], độ dài 3 -> nums[2] = 2
t =   2, idx = 0: t mod 6 = 2, co: nums[2:], độ dài 1 -> nums[2] = 2
t =   3, idx = 2: t mod 6 = 3, giãn: nums[:0], độ dài 0 -> -1 (vượt độ dài)
t =   5, idx = 0: t mod 6 = 5, giãn: nums[:2], độ dài 2 -> nums[0] = 0
t =   6, idx = 2: t mod 6 = 0, co: nums[0:], độ dài 3 -> nums[2] = 2
t = 100000, idx = 1: t mod 6 = 4, giãn: nums[:1], độ dài 1 -> -1 (vượt độ dài)


In [10]:
def trace_simulation(nums, minutes):
    n = len(nums); d = deque(nums); removed = deque(); shrinking = True
    print(f"phút 0: {list(d)}")
    for m in range(1, minutes + 1):
        if shrinking:
            removed.append(d.popleft())
            if not d: shrinking = False
        else:
            d.append(removed.popleft())
            if len(d) == n: shrinking = True
        lo, hi = segment(n, m)
        print(f"phút {m}: {list(d)}  | đoạn công thức nums[{lo}:{hi}] = {nums[lo:hi]}")

trace_simulation([7, 8, 9, 10], 9)

phút 0: [7, 8, 9, 10]
phút 1: [8, 9, 10]  | đoạn công thức nums[1:4] = [8, 9, 10]
phút 2: [9, 10]  | đoạn công thức nums[2:4] = [9, 10]
phút 3: [10]  | đoạn công thức nums[3:4] = [10]
phút 4: []  | đoạn công thức nums[0:0] = []
phút 5: [7]  | đoạn công thức nums[0:1] = [7]
phút 6: [7, 8]  | đoạn công thức nums[0:2] = [7, 8]
phút 7: [7, 8, 9]  | đoạn công thức nums[0:3] = [7, 8, 9]
phút 8: [7, 8, 9, 10]  | đoạn công thức nums[0:4] = [7, 8, 9, 10]
phút 9: [8, 9, 10]  | đoạn công thức nums[1:4] = [8, 9, 10]


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def elementInNums(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        n = len(nums)
        ans = []
        for t, i in queries:
            t %= 2 * n
            if t < n:                       # đã xóa t phần tử đầu
                ans.append(nums[t + i] if i < n - t else -1)
            else:                           # đã nối lại t - n phần tử đầu
                ans.append(nums[i] if i < t - n else -1)
        return ans

sol = Solution()
for (nums, qs), want in examples:
    got = sol.elementInNums(nums, qs)
    print(got, "OK" if got == want else "SAI")

[2, 2, -1, 0] OK
[2, -1, 2, -1] OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (trạng thái trong một chu kỳ).** Với `0 ≤ t ≤ n`: mảng ở phút `t` là `nums[t:]`. Với `n ≤ t ≤ 2n`: mảng là `nums[:t − n]`.
- Quy nạp pha co: phút 0 là `nums[0:]`; nếu phút `t < n` là `nums[t:]` (không rỗng) thì xóa trái nhất cho `nums[t+1:]`.
- Ở phút `n` mảng rỗng `= nums[n:] = nums[:0]`. Thứ tự đã xóa là `nums[0], nums[1], …, nums[n−1]`.
- Quy nạp pha giãn: nếu phút `t` (`n ≤ t < 2n`) là `nums[:t−n]` thì nối phần tử bị xóa tiếp theo `nums[t−n]` cho `nums[:t−n+1]`.

**Bổ đề 2 (chu kỳ).** Phút `2n` mảng là `nums[:n] = nums` — giống phút 0, và luật chuyển chỉ phụ thuộc trạng thái (cùng pha "co" vì mảng đầy) ⇒ tuần hoàn chu kỳ `2n`.

Kết hợp: phần tử chỉ số `idx` ở phút `t` là phần tử `idx` của `nums[t':]` hoặc `nums[:t'−n]` với `t' = t mod 2n` — đúng công thức, kể cả trường hợp rỗng (`t' = n`: độ dài `0` ⇒ `-1`). ✓

## 11. Phân tích Time / Space

`n ≤ 100`, `q ≤ 10⁵`, `time ≤ 10⁵`.

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. Mô phỏng | — | `O(time + n)` | `O(n)` |
| 2. Lưu một chu kỳ | `O(n²)` | `O(1)` | `O(n²)` |
| 3. Công thức | — | `O(1)` | `O(1)` |
| 4. Đoạn `[lo, hi)` | — | `O(1)` | `O(1)` |

In [12]:
rng = random.Random(1)
nums_big = [rng.randint(0, 100) for _ in range(100)]
qs_big = [[rng.randint(0, 10**5), rng.randrange(100)] for _ in range(10**5)]
res = []
for name, fn in [("precompute", element_precompute), ("formula", element_formula), ("segment", element_segment), ("submit", Solution().elementInNums)]:
    t0 = _time.perf_counter(); r = fn(nums_big, qs_big); res.append(r)
    print(f"{name:10s}: {(_time.perf_counter() - t0) * 1000:.1f} ms")
assert all(r == res[0] for r in res)
t0 = _time.perf_counter(); element_simulate(nums_big, qs_big[:200]); print(f"mô phỏng: 200 truy vấn mất {_time.perf_counter() - t0:.2f}s")

precompute: 9.6 ms
formula   : 8.1 ms
segment   : 14.8 ms
submit    : 8.3 ms


mô phỏng: 200 truy vấn mất 0.35s


## 12. Bộ test

Kiểm tra công thức với **mọi** `t` trong ba chu kỳ và mọi `idx`, đối chiếu mô phỏng liên tục (một lần chạy, ghi trạng thái từng phút).

In [13]:
fns = [element_precompute, element_formula, element_segment, Solution().elementInNums]

for (nums, qs), want in examples:
    assert element_simulate(nums, qs) == want
    for fn in fns:
        assert fn(nums, qs) == want

def all_states(nums, minutes):
    n = len(nums); d = deque(nums); removed = deque(); shrinking = True; st = [list(d)]
    for _ in range(minutes):
        if shrinking:
            removed.append(d.popleft())
            if not d: shrinking = False
        else:
            d.append(removed.popleft())
            if len(d) == n: shrinking = True
        st.append(list(d))
    return st

rng = random.Random(2024)
for _ in range(200):
    n = rng.randint(1, 8); nums = [rng.randint(0, 100) for _ in range(n)]
    states = all_states(nums, 6 * n + 3)
    qs = [[t, i] for t in range(len(states)) for i in range(n)]
    want = [states[t][i] if i < len(states[t]) else -1 for t, i in qs]
    for fn in fns:
        assert fn(nums, qs) == want, (fn, nums)
    assert element_simulate(nums, qs[:40]) == want[:40]
print("ví dụ + toàn bộ (t, idx) trong 3 chu kỳ của 200 mảng ngẫu nhiên khớp mô phỏng")

ví dụ + toàn bộ (t, idx) trong 3 chu kỳ của 200 mảng ngẫu nhiên khớp mô phỏng


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng chu kỳ `n` thay vì `2n`

In [14]:
def wrong_period_n(nums, queries):
    n = len(nums); out = []
    for t, i in queries:
        tp = t % n                                                    # SAI: chu kỳ là 2n
        out.append(nums[tp + i] if i < n - tp else -1)
    return out

for (nums, qs), want in examples:
    print(f"sai = {wrong_period_n(nums, qs)} | đúng = {want}")

sai = [2, 2, 2, 2] | đúng = [2, 2, -1, 0]
sai = [2, 2, 2, 2] | đúng = [2, -1, 2, -1]


Chu kỳ gồm `n` phút co **và** `n` phút giãn. Ví dụ 2 (`n = 1`): `t mod 1 = 0` luôn ⇒ mảng luôn đầy — sai ở phút lẻ. **Sửa:** `t mod 2n`.

### Lỗi 2 — Pha giãn lấy `nums[t' − n + idx]` (nhầm sang hậu tố)

In [15]:
def wrong_grow_offset(nums, queries):
    n = len(nums); out = []
    for t, i in queries:
        tp = t % (2 * n)
        if tp < n:
            out.append(nums[tp + i] if i < n - tp else -1)
        else:
            k = tp - n
            out.append(nums[(k + i) % n] if i < k else -1)            # SAI: pha giãn là TIỀN TỐ nums[:k]
    return out

for (nums, qs), want in examples + [(([7, 8, 9], [[4, 0], [5, 0], [5, 1]]), [7, 7, 8])]:
    print(f"sai = {wrong_grow_offset(nums, qs)} | đúng = {want}")

sai = [2, 2, -1, 2] | đúng = [2, 2, -1, 0]
sai = [2, -1, 2, -1] | đúng = [2, -1, 2, -1]
sai = [8, 9, 7] | đúng = [7, 7, 8]


Phần tử được nối lại theo thứ tự bị xóa: `nums[0]` trước ⇒ mảng giãn là `[nums[0], nums[1], …]`. **Sửa:** `nums[idx]`.

### Lỗi 3 — Xử lý sai phút `t' = n` (mảng rỗng)

In [16]:
def wrong_empty(nums, queries):
    n = len(nums); out = []
    for t, i in queries:
        tp = t % (2 * n)
        if tp <= n:                                                   # SAI ở biên: tp = n coi là pha co
            out.append(nums[min(tp + i, n - 1)] if i < max(1, n - tp) else -1)
        else:
            out.append(nums[i] if i < tp - n else -1)
    return out

for (nums, qs), want in examples + [(([0, 1, 2], [[3, 0], [9, 0]]), [-1, -1])]:
    print(f"sai = {wrong_empty(nums, qs)} | đúng = {want}")

sai = [2, 2, -1, 0] | đúng = [2, 2, -1, 0]
sai = [2, 2, 2, 2] | đúng = [2, -1, 2, -1]
sai = [2, 2] | đúng = [-1, -1]


Ở phút `n` (và `3n`, `5n`, …) mảng **rỗng** ⇒ mọi truy vấn `-1`. Code "vá" bằng `max(1, …)`/`min(…)` để tránh lỗi chỉ số sẽ trả về giá trị bịa. **Sửa:** để công thức tự nhiên: `t' = n` rơi vào nhánh giãn với độ dài `0`.

### Lỗi 4 — So sánh `idx ≤ độ dài` (lệch một)

In [17]:
def wrong_le_length(nums, queries):
    n = len(nums); out = []
    for t, i in queries:
        tp = t % (2 * n)
        L, src = (n - tp, tp + i) if tp < n else (tp - n, i)
        out.append(nums[src] if i <= L and src < n else -1)            # SAI: phải là i < L
    return out

for (nums, qs), want in examples + [(([0, 1, 2], [[4, 1], [1, 2]]), [-1, -1])]:
    print(f"sai = {wrong_le_length(nums, qs)} | đúng = {want}")

sai = [2, 2, -1, 0] | đúng = [2, 2, -1, 0]
sai = [2, 2, 2, 2] | đúng = [2, -1, 2, -1]
sai = [1, -1] | đúng = [-1, -1]


Mảng độ dài `L` có chỉ số `0..L−1`. Phút 4 của `[0,1,2]` là `[0]` ⇒ chỉ số 1 không tồn tại; điều kiện `≤` đọc nhầm `nums[1]`. **Sửa:** `idx < L`.

### Lỗi 5 — Mô phỏng nhưng xác định pha bằng độ dài (`len(d) > 0` ⇒ xóa)

Nếu luật là "còn phần tử thì xóa", sau khi nối lại một phần tử ở pha giãn ta lại xóa nó ngay ⇒ mảng dao động `[] ↔ [x]` mãi. Phải nhớ **pha** (biến `shrinking`) như Cách 1.

In [18]:
def wrong_phase_by_length(nums, minutes):
    d = deque(nums); removed = deque(); seq = [list(d)]
    for _ in range(minutes):
        if d:                                                         # SAI: "còn phần tử thì xóa"
            removed.append(d.popleft())
        else:
            d.append(removed.popleft())
        seq.append(list(d))
    return seq

print("sai :", wrong_phase_by_length([0, 1, 2], 8))
print("đúng:", all_states([0, 1, 2], 8))

sai : [[0, 1, 2], [1, 2], [2], [], [0], [], [1], [], [2]]
đúng: [[0, 1, 2], [1, 2], [2], [], [0], [0, 1], [0, 1, 2], [1, 2], [2]]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Quá trình lặp lại ⇒ tìm **chu kỳ** và dùng `t mod P`.
- Thao tác "xóa đầu / nối cuối theo thứ tự" ⇒ trạng thái luôn là **đoạn liên tiếp** của mảng gốc; mô tả bằng `(lo, hi)`.
- Khi chưa chắc công thức: **mô phỏng một chu kỳ** và lưu lại (Cách 2), rồi so sánh.
- Luôn kiểm tra trạng thái rỗng và chỉ số sát biên.
- Họ bài: 189 *Rotate Array*, 1823 *Find the Winner of the Circular Game*, 2073 *Time Needed to Buy Tickets*, 2582 *Pass the Pillow* (tuần hoàn chu kỳ `2(n−1)`).

## 15. Biến thể của bài

1. **Xóa từ phải** (phần tử phải nhất) rồi nối lại vào **đầu**: pha co là tiền tố `nums[:n − t']`, pha giãn là hậu tố.
2. **Xoay vòng** thay vì xóa/nối (đưa phần tử đầu xuống cuối mỗi phút): chu kỳ `n`, phần tử `nums[(t + idx) mod n]`.
3. **2582 Pass the Pillow:** gối chuyền qua lại `1 → n → 1`, chu kỳ `2(n − 1)`.
4. **Truy vấn tổng đoạn** tại phút `t`: kết hợp đoạn `(lo, hi)` với prefix sum.

In [19]:
def remove_right_variant(nums, queries):
    """Biến thể 1: xóa phải nhất, rồi nối lại vào ĐẦU theo đúng thứ tự bị xóa."""
    n = len(nums); out = []
    for t, i in queries:
        tp = t % (2 * n)
        lo, hi = (0, n - tp) if tp < n else (n - (tp - n), n)
        out.append(nums[lo + i] if lo + i < hi else -1)
    return out

def remove_right_sim(nums, minutes):
    n = len(nums); d = deque(nums); removed = deque(); shrinking = True; st = [list(d)]
    for _ in range(minutes):
        if shrinking:
            removed.append(d.pop())
            if not d: shrinking = False
        else:
            d.appendleft(removed.popleft())              # phần tử bị xóa trước (nums[n-1]) được nối trước
            if len(d) == n: shrinking = True
        st.append(list(d))
    return st

def pass_the_pillow(n, time):
    """Biến thể 3 (2582)."""
    k, r = divmod(time, n - 1)
    return r + 1 if k % 2 == 0 else n - r

rng = random.Random(15)
for _ in range(100):
    n = rng.randint(1, 6); nums = [rng.randint(0, 9) for _ in range(n)]
    st = remove_right_sim(nums, 4 * n)
    qs = [[t, i] for t in range(len(st)) for i in range(n)]
    assert remove_right_variant(nums, qs) == [st[t][i] if i < len(st[t]) else -1 for t, i in qs]
print("biến thể xóa phải khớp mô phỏng; 2582 ví dụ (n = 4, time = 5) ->", pass_the_pillow(4, 5), "(kỳ vọng 2)")

biến thể xóa phải khớp mô phỏng; 2582 ví dụ (n = 4, time = 5) -> 2 (kỳ vọng 2)


Ghi chú biến thể 1: pha co xóa `nums[n−1], nums[n−2], …`; pha giãn nối lại vào đầu theo đúng thứ tự đó ⇒ mảng giãn là **hậu tố** `nums[n − k:]` và sau `n` phút trở về `nums`. Đây là luật tự đặt; đổi luật (ví dụ nối theo thứ tự ngược) thì mảng có thể bị đảo và mất tính tuần hoàn chu kỳ `2n` — phải mô phỏng lại để kiểm chứng.

## 16. So sánh các cách

| Cách | Ý tưởng | Mỗi truy vấn | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | nghĩa đen | `O(time + n)` | `O(n)` | kiểm chứng |
| 2. Lưu một chu kỳ | tuần hoàn | `O(1)` | `O(n²)` | luật phức tạp, `n` nhỏ |
| 3. Công thức | hai pha | `O(1)` | `O(1)` | **nộp bài** |
| 4. Đoạn `[lo, hi)` | thống nhất | `O(1)` | `O(1)` | mở rộng biến thể |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [5, 6, 7, 8]`, các truy vấn `[1, 2]`, `[4, 0]`, `[6, 1]`, `[8, 3]`, `[13, 0]`?

**Bài 2 (tính tay):** với `n = 4`, viết `(lo, hi)` cho `t = 0..8`.

**Bài 3 (điền TODO):** hoàn thiện `length_at(n, t)` trả về độ dài mảng ở phút `t`.

**Bài 4 (tự viết lại):** viết `sum_at(nums, t)` trả về tổng các phần tử của mảng ở phút `t` trong `O(1)` sau khi dựng prefix sum.

**Bài 5 (mini challenge):** quá trình mới: mỗi phút xóa **hai** phần tử trái nhất (hoặc một nếu chỉ còn một) cho tới khi rỗng, rồi mỗi phút nối lại **một** phần tử theo thứ tự đã xóa. Trả lời truy vấn `[time, index]` trong `O(1)`. Tìm chu kỳ và công thức, kiểm chứng bằng mô phỏng. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [20]:
def length_at(n, t):
    # TODO: t' = t % (2n); co -> n - t'; giãn -> t' - n
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `2n = 8`. (2) `[1,2]`: co, `nums[1:] = [6,7,8]` ⇒ chỉ số 2 là `8`; `[4,0]`: rỗng ⇒ `-1`; `[6,1]`: giãn `nums[:2] = [5,6]` ⇒ `6`. (3) `[8,3]`: `t' = 0` ⇒ `8`; `[13,0]`: `t' = 5` ⇒ giãn `nums[:1]` ⇒ `5`.

**Bài 2:** (1) Co: `(0,4)`, `(1,4)`, `(2,4)`, `(3,4)`. (2) `t = 4`: `(0,0)`. (3) Giãn: `(0,1)`, `(0,2)`, `(0,3)`, rồi `t = 8` ≡ `0`: `(0,4)`.

**Bài 3:** (1) Một dòng với `if`. (2) Hoặc `hi − lo` của `segment`. (3) Kiểm tra bằng mô phỏng.

**Bài 4:** (1) `P[i]` = tổng `nums[:i]`. (2) Tổng đoạn `[lo, hi)` = `P[hi] − P[lo]`. (3) Kết hợp với `segment`.

**Bài 5:** (1) Pha co dài `⌈n/2⌉` phút: sau `t` phút (`t < ⌈n/2⌉`) mảng là `nums[2t:]`. (2) Pha giãn dài `n` phút: sau `k` phút giãn mảng là `nums[:k]`. (3) Chu kỳ `P = ⌈n/2⌉ + n`.

## 19. Đáp án tham khảo

In [21]:
# Bài 1
assert Solution().elementInNums([5, 6, 7, 8], [[1, 2], [4, 0], [6, 1], [8, 3], [13, 0]]) == [8, -1, 6, 8, 5]
# Bài 2
print("Bài 2:", [segment(4, t) for t in range(9)])
assert [segment(4, t) for t in range(9)] == [(0, 4), (1, 4), (2, 4), (3, 4), (0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]

# Bài 3
def length_at_ans(n, t):
    tp = t % (2 * n)
    return n - tp if tp < n else tp - n

# Bài 4
from itertools import accumulate
def sum_at_factory(nums):
    P = [0, *accumulate(nums)]; n = len(nums)
    def sum_at(t):
        lo, hi = segment(n, t); return P[hi] - P[lo]
    return sum_at

rng = random.Random(41)
for _ in range(200):
    n = rng.randint(1, 8); nums = [rng.randint(0, 50) for _ in range(n)]
    st = all_states(nums, 5 * n); f = sum_at_factory(nums)
    for t, s in enumerate(st):
        assert length_at_ans(n, t) == len(s) and f(t) == sum(s)
print("Bài 1-4: OK")

Bài 2: [(0, 4), (1, 4), (2, 4), (3, 4), (0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]
Bài 1-4: OK


In [22]:
# Bài 5 — xóa hai, nối một
def two_remove_states(nums, minutes):
    n = len(nums); d = deque(nums); removed = deque(); shrinking = True; st = [list(d)]
    for _ in range(minutes):
        if shrinking:
            for _ in range(2):
                if d: removed.append(d.popleft())
            if not d: shrinking = False
        else:
            d.append(removed.popleft())
            if len(d) == n: shrinking = True
        st.append(list(d))
    return st

def two_remove_query(nums, queries):
    n = len(nums); S = (n + 1) // 2; P = S + n; out = []
    for t, i in queries:
        tp = t % P
        if tp < S:
            lo, hi = 2 * tp, n                     # đã xóa 2*tp phần tử
        else:
            lo, hi = 0, tp - S                     # đã nối lại tp - S phần tử
        out.append(nums[lo + i] if lo + i < hi else -1)
    return out

rng = random.Random(55)
for _ in range(200):
    n = rng.randint(1, 9); nums = [rng.randint(0, 9) for _ in range(n)]
    st = two_remove_states(nums, 4 * n + 5)
    qs = [[t, i] for t in range(len(st)) for i in range(n)]
    assert two_remove_query(nums, qs) == [st[t][i] if i < len(st[t]) else -1 for t, i in qs], nums
print("Bài 5: công thức chu kỳ ceil(n/2) + n khớp mô phỏng; trạng thái n = 5:", two_remove_states([1, 2, 3, 4, 5], 8))

Bài 5: công thức chu kỳ ceil(n/2) + n khớp mô phỏng; trạng thái n = 5: [[1, 2, 3, 4, 5], [3, 4, 5], [5], [], [1], [1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5]]


## 20. Template nhớ nhanh + Checklist

```python
# Quá trình: xóa trái nhất tới rỗng, rồi nối lại theo thứ tự -> chu kỳ 2n
t %= 2 * n
if t < n:     ans = nums[t + i] if i < n - t else -1     # mảng = nums[t:]
else:         ans = nums[i]     if i < t - n else -1     # mảng = nums[:t - n]
```

**Checklist trước khi submit:**
- [ ] Chu kỳ `2n`, không phải `n`.
- [ ] Pha co là **hậu tố** `nums[t:]`, pha giãn là **tiền tố** `nums[:t−n]`.
- [ ] `t mod 2n = n` ⇒ mảng rỗng ⇒ `-1`.
- [ ] So sánh `idx < độ dài` (ngặt).
- [ ] Mô phỏng (nếu dùng) phải nhớ **pha**, không suy pha từ độ dài.